# Phase 3 — Options exotiques

Ce notebook documente le livrable Phase 3 : trois types d'options exotiques pricés et validés.

## Livrable

```text
3 types d'options exotiques pricés et validés
```

Options retenues :

1. Down-and-Out barrier call.
2. Asian arithmetic call.
3. Lookback floating-strike call.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'neuroprice').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

PROJECT_ROOT


## 1. Down-and-Out barrier call

Payoff terminal avec barrière absorbante :

```text
V(B, tau) = 0
V(S, 0) = max(S - K, 0) pour S > B
```

Validation : formule semi-analytique par méthode des images.

Commande de reproduction :

```bash
python -m scripts.train_pinn_barrier_down_out --pretrain-epochs 3000 --epochs 8000 --hidden-dim 128 --hidden-layers 5 --output-transform barrier --n-interior 8192 --n-terminal 4096 --n-boundary 4096 --n-supervised 8192 --supervised-weight 0.1 --lbfgs-steps 500 --out-dir artifacts/phase3_down_out_barrier
python -m scripts.validate_pinn_barrier_down_out --checkpoint artifacts/phase3_down_out_barrier/down_out_barrier_pinn.pt --out artifacts/phase3_down_out_barrier/benchmark.json --n-points 10000 --relative-floor 1.0 --min-reference-price 1.0
```

Résultat retenu :

```text
relative_l2 = 0.2957%
tradable_p95_point_relative_error = 9.8904%
tradable_pct_points_under_5pct_error = 93.43%
pinn_points_per_second = 474,467
```


## 2. Asian arithmetic call

Payoff basé sur la moyenne arithmétique discrète :

```text
payoff = max(A_T - K, 0)
```

Le PINN 3D `(S, A, tau)` a été conservé comme piste expérimentale. Le jalon validé utilise un surrogate supervisé Monte Carlo offline sur le cas standard `A0 = S0`.

Commande validée :

```bash
python -m scripts.train_asian_surrogate_offline --epochs 3000 --dataset-samples 20000 --batch-size 512 --mc-paths 20000 --mc-steps 64 --mc-chunk-size 2000 --hidden-dim 128 --hidden-layers 5 --out-dir artifacts/phase3_asian_surrogate_offline
python -m scripts.validate_asian_surrogate --checkpoint artifacts/phase3_asian_surrogate_offline/asian_surrogate.pt --out artifacts/phase3_asian_surrogate_offline/benchmark.json --n-points 1000 --mc-paths 20000 --mc-steps 64 --mc-chunk-size 2000 --relative-floor 1.0 --min-reference-price 1.0
```

Résultat validé :

```text
relative_l2 = 0.16%
tradable_p95_point_relative_error = 1.39%
tradable_pct_points_under_5pct_error = 99.11%
tradable_pct_points_under_10pct_error = 99.85%
pinn_vs_monte_carlo_speedup = 4680.98x
```


## 3. Lookback floating-strike call

Payoff dépendant du minimum réalisé du sous-jacent :

```text
payoff = max(S_T - min(S_t), 0)
```

Validation : surrogate supervisé Monte Carlo offline sur le cas initial standard `min_0 = S0`.

Commande validée :

```bash
python -m scripts.train_lookback_surrogate_offline --epochs 3000 --dataset-samples 20000 --batch-size 512 --mc-paths 20000 --mc-steps 64 --mc-chunk-size 2000 --hidden-dim 128 --hidden-layers 5 --out-dir artifacts/phase3_lookback_surrogate_offline
python -m scripts.validate_lookback_surrogate --checkpoint artifacts/phase3_lookback_surrogate_offline/lookback_surrogate.pt --out artifacts/phase3_lookback_surrogate_offline/benchmark.json --n-points 1000 --mc-paths 20000 --mc-steps 64 --mc-chunk-size 2000 --relative-floor 1.0 --min-reference-price 1.0
```

Résultat validé :

```text
relative_l2 = 0.66%
tradable_p95_point_relative_error = 1.48%
tradable_pct_points_under_5pct_error = 99.48%
tradable_pct_points_under_10pct_error = 99.69%
pinn_vs_monte_carlo_speedup = 5475.93x
```


## Chargement des benchmarks JSON

Cette cellule agrège les benchmarks existants dans `artifacts/`.


In [ ]:
benchmark_files = {
    'down_out_barrier': PROJECT_ROOT / 'artifacts' / 'phase3_down_out_barrier' / 'benchmark.json',
    'asian_arithmetic_offline': PROJECT_ROOT / 'artifacts' / 'phase3_asian_surrogate_offline' / 'benchmark.json',
    'lookback_floating_offline': PROJECT_ROOT / 'artifacts' / 'phase3_lookback_surrogate_offline' / 'benchmark.json',
}

rows = []
for name, path in benchmark_files.items():
    if not path.exists():
        rows.append({'option': name, 'status': 'missing benchmark'})
        continue
    data = json.loads(path.read_text(encoding='utf-8'))
    rows.append({
        'option': name,
        'status': 'loaded',
        'relative_l2': data.get('relative_l2'),
        'tradable_p95_point_relative_error': data.get('tradable_p95_point_relative_error'),
        'tradable_pct_under_5pct': data.get('tradable_pct_points_under_5pct_error'),
        'tradable_pct_under_10pct': data.get('tradable_pct_points_under_10pct_error'),
        'max_absolute_error': data.get('max_absolute_error'),
        'pinn_points_per_second': data.get('pinn_points_per_second'),
        'speedup_vs_mc': data.get('pinn_vs_monte_carlo_speedup'),
    })

benchmarks = pd.DataFrame(rows)
benchmarks


## Validation du livrable

Critères Phase 3 suivis pour les exotiques Monte Carlo :

```text
relative_l2 < 10%
tradable_p95_point_relative_error < 25%
tradable_pct_points_under_10pct_error > 70%
PINN/surrogate au moins 50x plus rapide que Monte Carlo
```

La barrière utilise une référence semi-analytique et un critère plus strict sur la zone tradable.


In [ ]:
if 'benchmarks' in globals():
    loaded = benchmarks[benchmarks['status'] == 'loaded'].copy()
    summary = {
        'n_loaded_benchmarks': int(len(loaded)),
        'phase3_deliverable_complete': int(len(loaded)) == 3,
    }
    summary
else:
    {}


## Conclusion Phase 3

La Phase 3 est clôturée :

| Option | Validation | Statut |
| --- | --- | --- |
| Down-and-Out barrier call | Semi-analytique | Validée |
| Asian arithmetic call | Monte Carlo haute précision | Validée |
| Lookback floating-strike call | Monte Carlo haute précision | Validée |

Le projet peut passer à la Phase 4 : API & Backend.
